# Prototipado y Afinamiento de AutoML con TPOT

Este notebook documenta el proceso de experimentación y estudio previo a la integración de **TPOT** en el pipeline de producción de Kedro. El objetivo es validar la configuración genética de AutoML para asegurar que el modelo pueda converger eficientemente con los datos de ventas disponibles.

In [ ]:
import pandas as pd
import numpy as np
from tpot import TPOTClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

## 1. Carga de Datos Integrados
Cargamos el dataset maestro que ya ha sido generado por el pipeline de integración. Usamos la ruta directa al archivo CSV para asegurar la compatibilidad.

In [ ]:
# Cargamos el dataset maestro generado por Kedro en la capa 03_primary
try:
    # Intentamos cargar vía catalog si el entorno Kedro está activo
    df = catalog.load("dataset_final")
except NameError:
    # Si catalog no existe (ej. corriendo fuera de kedro jupyter), cargamos el archivo directamente
    df = pd.read_csv('../data/03_primary/dataset_final.csv')

df.head()

## 2. Preparación de Variables para Clasificación
El objetivo es predecir el `segmento` del cliente. Como TPOT requiere datos numéricos para procesar los algoritmos de Scikit-Learn, aplicamos `LabelEncoder` al target.

In [ ]:
target = 'segmento'
df_ml = df.dropna(subset=[target]).copy()

# Codificamos la variable objetivo (VIP, Premium, etc -> 0, 1, 2)
le = LabelEncoder()
y = le.fit_transform(df_ml[target].astype(str))

# Seleccionamos solo variables numéricas como predictores
X = df_ml.select_dtypes(include=[np.number]).fillna(0)

# División para validación
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Configuración de TPOT (Optimización Genética)
TPOT no es un modelo, es un **Optimizador**. Utiliza algoritmos genéticos para probar miles de tuberías (pipelines) de preprocesamiento y modelos. 

**Parámetros clave estudiados:**
- `generations`: Cuántas veces evolucionará la población de modelos (similar a épocas en redes neuronales).
- `population_size`: Cuántos modelos diferentes se crean en cada generación.
- `cv`: Validación cruzada para asegurar que el modelo no aprenda los datos de memoria (Overfitting).
- `scoring`: La métrica que TPOT intentará maximizar (en este caso, Accuracy).

In [ ]:
# Inicializamos TPOT con una configuración controlada para el prototipo
tpot = TPOTClassifier(
    generations=3,         # Evolucionamos 3 veces
    population_size=12,    # Cada generación tiene 12 individuos (modelos)
    cv=3,                  # Usamos 3 folds para validación cruzada
    random_state=42,
    verbosity=2,           # Muestra el progreso en tiempo real
    max_time_mins=5        # Límite de seguridad para no exceder recursos
)

tpot.fit(X_train, y_train)

## 4. Evaluación de Resultados
Una vez que TPOT encuentra el "modelo ganador", lo evaluamos con el set de pruebas que el algoritmo nunca ha visto.

In [ ]:
score = tpot.score(X_test, y_test)
print(f"Accuracy final del prototipo: {score:.4f}")

# Mostramos el pipeline exacto que TPOT decidió que era el mejor
print("\n--- Pipeline Ganador ---")
print(tpot.fitted_pipeline_)

## 5. Exportación para Producción
TPOT permite exportar el código de Scikit-Learn directamente. Este código es la base de lo que luego integramos de forma modular en `nodes.py` dentro de Kedro.

In [ ]:
# Guardamos el código generado por si necesitamos auditoría manual
tpot.export('prototipo_tpot_pipeline.py')